# TrendPulse — Part 1: Python Foundations

**Goal:** Load trending-video CSV data for the United States, Great Britain, and India, then use core Python for cleaning, data structures, loops, and rule-based classification.

**Pandas boundary:** This notebook uses Pandas only for `pd.read_csv()` and `df.to_dict("records")` to load data into Python lists of dictionaries. All cleaning, counting, grouping, and classification after loading use core Python.

## Notebook discipline
- Sections use clear Markdown headings and explanatory notes.
- Run from the first cell to the last cell using **Runtime → Restart session and run all**.
- No cell relies on a hidden variable created by running cells out of order.
- The notebook prints counts and parse-failure reports so results can be checked.

## 1. Setup and configuration

The notebook expects these files in `part1_python_foundations/data/`:

- `USvideos.csv`
- `GBvideos.csv`
- `INvideos.csv`
- `categories.csv`

If the `data/` folder is not available in the Colab runtime, the next cell will ask you to upload all four CSV files. Keep the filenames as listed above.

In [1]:
from pathlib import Path
from datetime import datetime, date
from collections import Counter
import math
import pandas as pd

DATA_DIR = Path("data")
COUNTRIES = {
    "US": "USvideos.csv",
    "GB": "GBvideos.csv",
    "IN": "INvideos.csv",
}
CATEGORIES_FILE = "categories.csv"

# Rule-based classification thresholds (documented here for reproducibility)
VIRAL_VIEWS_THRESHOLD = 1_000_000
STEADY_VIEWS_THRESHOLD = 100_000
MIN_DIVISIVE_RATINGS = 1_000
VIRAL_LIKE_RATIO = 0.80
STEADY_LIKE_RATIO = 0.70
DIVISIVE_LIKE_RATIO = 0.60

run_started_at = datetime.now().astimezone()
print("=" * 68)
print("TrendPulse | Part 1: Python Foundations")
print(f"Run started: {run_started_at:%Y-%m-%d %H:%M:%S %Z}")
print(f"Countries to load: {', '.join(COUNTRIES.keys())}")
print(f"Data directory: {DATA_DIR.resolve()}")
print("=" * 68)

TrendPulse | Part 1: Python Foundations
Run started: 2026-10-27 11:07:28 UTC
Countries to load: US, GB, IN
Data directory: /content/data


### Make the data available

In Colab, upload the four CSV files if they are not already in the `data/` folder. Uploads are copied into `data/` so the remaining cells use consistent paths.

In [2]:
required_filenames = list(COUNTRIES.values()) + [CATEGORIES_FILE]
missing_files = [name for name in required_filenames if not (DATA_DIR / name).exists()]

if missing_files:
    try:
        from google.colab import files
        print("Please select these files from your computer:", ", ".join(missing_files))
        uploaded = files.upload()
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        for filename, file_bytes in uploaded.items():
            (DATA_DIR / filename).write_bytes(file_bytes)
    except ImportError as exc:
        raise FileNotFoundError(
            f"Missing files in {DATA_DIR.resolve()}: {missing_files}. "
            "Create the data folder and place the required CSV files there."
        ) from exc

missing_files = [name for name in required_filenames if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Still missing: {missing_files}. Check the uploaded filenames and try again."
    )

print("Files ready:")
for filename in required_filenames:
    print(f"- {filename}: {(DATA_DIR / filename).stat().st_size:,} bytes")

Files ready:
- USvideos.csv: 4,850,597 bytes
- GBvideos.csv: 4,575,850 bytes
- INvideos.csv: 4,544,641 bytes
- categories.csv: 292 bytes


## 2. Reusable loading function and category lookup

`load_country()` has parameters, a default argument (`max_rows=None`), and a return value. It validates the columns needed by later tasks, adds the country code, and returns a list of dictionaries.

In [3]:
REQUIRED_COLUMNS = {
    "video_id", "trending_date", "title", "channel_title", "category_id",
    "publish_time", "views", "likes", "dislikes", "comment_count",
    "comments_disabled", "ratings_disabled"
}

def load_country(path, country, max_rows=None):
    """Load one country CSV and return its rows as a list of dictionaries."""
    df = pd.read_csv(path, nrows=max_rows)
    missing_columns = REQUIRED_COLUMNS.difference(df.columns)
    if missing_columns:
        raise ValueError(
            f"{country} file is missing required columns: {sorted(missing_columns)}"
        )
    df["country"] = country
    records = df.to_dict("records")
    print(f"{country}: loaded {len(records):,} records from {Path(path).name}")
    return records

all_records = []
records_by_country = {}

for country, filename in COUNTRIES.items():
    country_records = load_country(DATA_DIR / filename, country)
    records_by_country[country] = country_records
    all_records.extend(country_records)

print(f"\nTotal loaded records: {len(all_records):,}")
print(f"Countries loaded: {len(records_by_country)}")

# Build category_id -> category name dictionary.
category_df = pd.read_csv(DATA_DIR / CATEGORIES_FILE)
if not {"category_id", "category_name"}.issubset(category_df.columns):
    raise ValueError("categories.csv must contain category_id and category_name.")

category_names = {
    str(row["category_id"]).strip(): str(row["category_name"]).strip()
    for row in category_df.to_dict("records")
}
print(f"Category names loaded: {len(category_names)}")
print("Sample category lookup:", list(category_names.items())[:5])

assert len(records_by_country) >= 3, "The assignment requires at least 3 countries."
assert len(all_records) >= 40_000, "The assignment requires at least 40,000 records."

US: loaded 16,080 records from USvideos.csv
GB: loaded 15,075 records from GBvideos.csv
IN: loaded 15,075 records from INvideos.csv

Total loaded records: 46,230
Countries loaded: 3
Category names loaded: 17
Sample category lookup: [('1', 'Film & Animation'), ('2', 'Autos & Vehicles'), ('10', 'Music'), ('15', 'Pets & Animals'), ('17', 'Sports')]


## 3. Type conversion and cleaning with core Python

**Cleaning decisions**
- Count fields (`views`, `likes`, `dislikes`, `comment_count`): convert to non-negative integers. Blank, malformed, or negative values are replaced with `0` so a bad count does not crash the analysis. We retain a flag on each record so classification can label records with missing/bad required metrics as `Unrated`.
- Boolean flags: accept common true/false strings and actual booleans. Missing or unrecognized values are replaced with `False`, and parse failures are counted.
- `trending_date`: the supplied YouTube dataset commonly uses `yy.dd.mm` (for example `17.14.11`). Parse with `%y.%d.%m`; invalid values become `None`.
- `publish_time`: parse the ISO-like timestamp with `datetime.fromisoformat`; invalid values become `None`.
- We do not drop a record just because one field is malformed. Keeping the row preserves country-level record counts, while the failure report makes data quality visible.
- This dataset has some CSV values that Pandas may interpret as missing (`NaN`); the helpers handle those explicitly.

In [4]:
NUMERIC_FIELDS = ("views", "likes", "dislikes", "comment_count")
BOOLEAN_FIELDS = (
    "comments_disabled", "ratings_disabled", "video_error_or_removed"
)

parse_failures = {
    field: 0 for field in NUMERIC_FIELDS + BOOLEAN_FIELDS + ("trending_date", "publish_time")
}
cleaned_records = []

def is_blank(value):
    """Return True for None, NaN, or whitespace-only strings."""
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    return isinstance(value, str) and value.strip() == ""

def parse_nonnegative_int(value, field_name):
    """Return (integer value, whether conversion failed)."""
    if is_blank(value):
        return 0, True
    try:
        # Handles values like 12, 12.0, and strings such as '1,234'.
        number = int(float(str(value).replace(",", "").strip()))
        if number < 0:
            return 0, True
        return number, False
    except (TypeError, ValueError, OverflowError):
        return 0, True

def parse_bool(value, field_name):
    """Return (boolean value, whether conversion failed)."""
    if isinstance(value, bool):
        return value, False
    if is_blank(value):
        return False, True
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes", "y"}:
        return True, False
    if normalized in {"false", "0", "no", "n"}:
        return False, False
    return False, True

def parse_trending_date(value):
    """Parse the compact yy.dd.mm trending date into datetime.date."""
    if is_blank(value):
        return None, True
    try:
        return datetime.strptime(str(value).strip(), "%y.%d.%m").date(), False
    except (TypeError, ValueError):
        return None, True

def parse_publish_time(value):
    """Parse an ISO-style publish timestamp into a datetime object."""
    if is_blank(value):
        return None, True
    try:
        text_value = str(value).strip()
        # fromisoformat accepts offsets; replace terminal Z for compatibility.
        if text_value.endswith("Z"):
            text_value = text_value[:-1] + "+00:00"
        return datetime.fromisoformat(text_value), False
    except (TypeError, ValueError):
        return None, True

for original in all_records:
    record = dict(original)  # copy the row so the raw records remain unchanged
    record["_missing_required_metric"] = False

    for field in NUMERIC_FIELDS:
        converted, failed = parse_nonnegative_int(record.get(field), field)
        record[field] = converted
        parse_failures[field] += int(failed)
        if failed:
            record["_missing_required_metric"] = True

    for field in BOOLEAN_FIELDS:
        # This dataset may not have every optional boolean column in every copy.
        if field in record:
            converted, failed = parse_bool(record.get(field), field)
            record[field] = converted
            parse_failures[field] += int(failed)
        else:
            record[field] = False

    parsed_date, failed = parse_trending_date(record.get("trending_date"))
    record["trending_date"] = parsed_date
    parse_failures["trending_date"] += int(failed)

    parsed_publish_time, failed = parse_publish_time(record.get("publish_time"))
    record["publish_time"] = parsed_publish_time
    parse_failures["publish_time"] += int(failed)

    record["video_id"] = "" if is_blank(record.get("video_id")) else str(record["video_id"]).strip()
    record["title"] = "" if is_blank(record.get("title")) else str(record["title"])
    record["channel_title"] = (
        "Unknown channel" if is_blank(record.get("channel_title"))
        else str(record["channel_title"]).strip()
    )
    record["category_id"] = "" if is_blank(record.get("category_id")) else str(record["category_id"]).strip()
    record["category_name"] = category_names.get(record["category_id"], "Unknown category")
    cleaned_records.append(record)

print("Parse-failure counts by field:")
for field, count in parse_failures.items():
    print(f"- {field}: {count:,}")
print(f"\nCleaned records retained: {len(cleaned_records):,}")

Parse-failure counts by field:
- views: 0
- likes: 12
- dislikes: 0
- comment_count: 36
- comments_disabled: 0
- ratings_disabled: 0
- video_error_or_removed: 0
- trending_date: 0
- publish_time: 0

Cleaned records retained: 46,230


## 4. Core data structures

- **List of dictionaries:** each row has named fields and is convenient for record-by-record cleaning and classification. Lists and dictionaries are mutable.
- **Tuples:** a compact snapshot of `video_id`, `trending_date`, `views`, and `likes`; tuples are immutable, so they suit a fixed record view.
- **Sets:** efficient unique-video comparisons; sets are mutable, but set operations return new sets.
- **Dictionary counter:** channel name → number of trending appearances, useful for frequency counts. Dictionaries are mutable.
- **Nested dictionary:** country → category name → record count; dictionaries are mutable and represent a hierarchy naturally.
- **Slicing:** `[:15]` selects the top 15 sorted channels without a manual counter loop.

In [5]:
# A tuple snapshot of selected fields (tuples are immutable).
video_snapshots = [
    (
        record["video_id"],
        record["trending_date"],
        record["views"],
        record["likes"],
    )
    for record in cleaned_records
]
print("Example tuple snapshot:", video_snapshots[0] if video_snapshots else None)

# Sets of unique video IDs for each country.
video_ids_by_country = {
    country: {r["video_id"] for r in country_records if r["video_id"]}
    for country, country_records in records_by_country.items()
}
for country, ids in video_ids_by_country.items():
    print(f"{country}: {len(ids):,} unique video IDs")

# Videos appearing in more than one country: pairwise intersections.
country_codes = list(video_ids_by_country)
cross_country_videos = set()
for i in range(len(country_codes)):
    for j in range(i + 1, len(country_codes)):
        left = country_codes[i]
        right = country_codes[j]
        shared = video_ids_by_country[left] & video_ids_by_country[right]
        cross_country_videos.update(shared)
        print(f"Shared between {left} and {right}: {len(shared):,}")

# Example set difference: IDs seen in US but not in GB or IN.
us_only = video_ids_by_country["US"] - (
    video_ids_by_country["GB"] | video_ids_by_country["IN"]
)
print(f"US-only video IDs: {len(us_only):,}")
print(f"Videos appearing in 2+ countries: {len(cross_country_videos):,}")

# Count appearances per channel using a dictionary.
channel_appearances = {}
for record in cleaned_records:
    channel = record["channel_title"]
    channel_appearances[channel] = channel_appearances.get(channel, 0) + 1

top_15_channels = sorted(
    channel_appearances.items(),
    key=lambda item: item[1],
    reverse=True
)[:15]  # slicing returns only the first 15 entries

print("\nTop 15 channels by trending appearances:")
for rank, (channel, appearances) in enumerate(top_15_channels, start=1):
    print(f"{rank:>2}. {channel}: {appearances:,}")

# Nested dictionary: country -> category name -> record count.
country_category_counts = {}
for record in cleaned_records:
    country = record["country"]
    category = record["category_name"]
    if country not in country_category_counts:
        country_category_counts[country] = {}
    country_category_counts[country][category] = (
        country_category_counts[country].get(category, 0) + 1
    )

print("\nCategory summary samples:")
for country, category_counts in country_category_counts.items():
    print(f"{country}: {list(category_counts.items())[:5]}")

Example tuple snapshot: ('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
US: 5,068 unique video IDs
GB: 4,113 unique video IDs
IN: 5,878 unique video IDs
Shared between US and GB: 134
Shared between US and IN: 120
Shared between GB and IN: 126
US-only video IDs: 4,834
Videos appearing in 2+ countries: 340

Top 15 channels by trending appearances:
 1. QuietChallenge75: 2,464
 2. OfficialReview41: 1,586
 3. UltimateUnboxing65: 1,433
 4. InsaneReaction46: 1,233
 5. Brand NewChallenge45: 1,175
 6. CrazyTrailer91: 1,018
 7. FinalTrailer17: 911
 8. UltimateSpeech20: 770
 9. Brand NewSecret51: 684
10. WildHighlights7: 682
11. SurprisingHighlights41: 650
12. FinalBreakdown11: 643
13. CrazyRecipe67: 603
14. QuietInterview9: 570
15. UltimateVlog96: 565

Category summary samples:
US: [('People & Blogs', 1423), ('News & Politics', 755), ('Music', 3052), ('Education', 561), ('Entertainment', 4371)]
GB: [('Science & Technology', 470), ('Howto & Style', 588), ('Comedy', 941), ('Entertainment

## 5. Variable scope

A variable created inside a function is local to that function. A variable defined outside is global to the notebook's module scope. The example below shows that a local variable cannot be accessed by its name outside the function.

In [6]:
scope_message = "I am defined outside the function."

def demonstrate_scope():
    local_message = "I am defined inside the function."
    print("Inside function, local:", local_message)
    print("Inside function, outer:", scope_message)
    return local_message

returned_message = demonstrate_scope()
print("Outside function, outer:", scope_message)
print("Outside function, returned local value:", returned_message)

try:
    print(local_message)  # NameError: local_message only exists inside the function
except NameError as error:
    print("Outside function, local variable is unavailable:", type(error).__name__)

Inside function, local: I am defined inside the function.
Inside function, outer: I am defined outside the function.
Outside function, outer: I am defined outside the function.
Outside function, returned local value: I am defined inside the function.
Outside function, local variable is unavailable: NameError


## 6. Loops over strings and sequences

`title_features()` uses:
- a `for` loop to inspect characters,
- `range()` with index access to inspect adjacent characters,
- word iteration to count uppercase words and word count.

The `while` loop below scans records until it finds the first record at or above the view threshold. `continue` skips records without a usable title, and `break` stops after the first match.

In [7]:
def title_features(title):
    """Count uppercase words, exclamation/question marks, and words in a title."""
    title = "" if is_blank(title) else str(title)
    words = title.split()
    uppercase_words = 0

    for word in words:
        # Count a word as uppercase if it contains a letter and all letters are uppercase.
        letters = [character for character in word if character.isalpha()]
        if letters and all(character.isupper() for character in letters):
            uppercase_words += 1

    punctuation_count = {"!": 0, "?": 0}
    for character in title:
        if character == "!":
            punctuation_count["!"] += 1
        elif character == "?":
            punctuation_count["?"] += 1

    # range() and index access: count adjacent repeated punctuation pairs as a small feature.
    repeated_punctuation_pairs = 0
    for index in range(len(title) - 1):
        if title[index] in "!?":
            if title[index] == title[index + 1]:
                repeated_punctuation_pairs += 1

    return {
        "uppercase_words": uppercase_words,
        "exclamation_marks": punctuation_count["!"],
        "question_marks": punctuation_count["?"],
        "word_count": len(words),
        "repeated_punctuation_pairs": repeated_punctuation_pairs,
    }

print(title_features("WOW!!! Is THIS REAL?"))

VIEW_SCAN_THRESHOLD = 1_000_000
index = 0
first_high_view_record = None

while index < len(cleaned_records):
    current_record = cleaned_records[index]
    index += 1

    if not current_record["title"].strip():
        continue  # no useful title; keep scanning

    if current_record["views"] >= VIEW_SCAN_THRESHOLD:
        first_high_view_record = current_record
        break  # stop once the first matching record is found

if first_high_view_record:
    print(
        "First record above view threshold:",
        first_high_view_record["country"],
        first_high_view_record["video_id"],
        first_high_view_record["views"],
        title_features(first_high_view_record["title"])
    )
else:
    print(f"No record found with at least {VIEW_SCAN_THRESHOLD:,} views.")

{'uppercase_words': 3, 'exclamation_marks': 3, 'question_marks': 1, 'word_count': 4, 'repeated_punctuation_pairs': 2}
First record above view threshold: US 3KkxacvDZi1 2841278 {'uppercase_words': 1, 'exclamation_marks': 0, 'question_marks': 0, 'word_count': 4, 'repeated_punctuation_pairs': 0}


## 7. Rule-based classification

The thresholds are configuration variables in Section 1:
- **Viral Hit:** at least 1,000,000 views and like ratio ≥ 0.80.
- **Steady Performer:** at least 100,000 views and like ratio ≥ 0.70, unless a higher-priority rule applies.
- **Divisive:** at least 1,000 total likes + dislikes and like ratio < 0.60.
- **Low Engagement:** remaining records with valid metrics.
- **Unrated:** ratings are disabled, required count fields were missing/bad before substitution, or the total likes + dislikes is zero.

The like ratio is `likes / (likes + dislikes)`. The denominator is checked before division. `comments_disabled` is handled as a separate flag: it does not by itself prevent a rating-based label because the label is based on views and likes/dislikes.

In [8]:
def classify_video(record):
    """Assign a rule-based label while safely handling missing/disabled ratings."""
    # Nested conditions handle the cases where a like ratio cannot be interpreted.
    if record.get("ratings_disabled", False):
        return "Unrated"
    else:
        if record.get("_missing_required_metric", False):
            return "Unrated"
        else:
            views = record.get("views", 0)
            likes = record.get("likes", 0)
            dislikes = record.get("dislikes", 0)
            total_ratings = likes + dislikes

            if total_ratings == 0:
                return "Unrated"

            like_ratio = likes / total_ratings

            if views >= VIRAL_VIEWS_THRESHOLD and like_ratio >= VIRAL_LIKE_RATIO:
                return "Viral Hit"
            elif total_ratings >= MIN_DIVISIVE_RATINGS and like_ratio < DIVISIVE_LIKE_RATIO:
                return "Divisive"
            elif views >= STEADY_VIEWS_THRESHOLD and like_ratio >= STEADY_LIKE_RATIO:
                return "Steady Performer"
            else:
                return "Low Engagement"

# Hand-trace example:
# views=1,500,000, likes=9,000, dislikes=1,000, ratings enabled, metrics valid.
# ratings_disabled is False; metrics are valid; total_ratings=10,000;
# like_ratio=9,000/10,000=0.90; views >= 1,000,000 and ratio >= 0.80,
# so the first classification branch returns "Viral Hit".
example_record = {
    "views": 1_500_000,
    "likes": 9_000,
    "dislikes": 1_000,
    "ratings_disabled": False,
    "comments_disabled": False,
    "_missing_required_metric": False,
}
print("Hand-traced example label:", classify_video(example_record))

label_counts = {}
for record in cleaned_records:
    label = classify_video(record)
    record["classification"] = label
    label_counts[label] = label_counts.get(label, 0) + 1

print("\nRecord counts by classification:")
for label in ("Viral Hit", "Steady Performer", "Divisive", "Low Engagement", "Unrated"):
    print(f"- {label}: {label_counts.get(label, 0):,}")
print("Total classified:", sum(label_counts.values()))

Hand-traced example label: Viral Hit

Record counts by classification:
- Viral Hit: 9,486
- Steady Performer: 26,892
- Divisive: 210
- Low Engagement: 9,050
- Unrated: 592
Total classified: 46230


## 8. Refactoring

Repeated data-quality handling was moved into reusable functions (`is_blank`, `parse_nonnegative_int`, `parse_bool`, `parse_trending_date`, and `parse_publish_time`). This avoids copying the same conversion and error-handling logic across fields. It also makes the cleaning policy easier to test and change in one place.

In [9]:
# Reusable summary function: used for quick country-level checks.
def summarize_country(records):
    """Return a small summary for a country's cleaned records."""
    total_views = sum(record["views"] for record in records)
    average_views = total_views / len(records) if records else 0
    unique_video_count = len({record["video_id"] for record in records if record["video_id"]})
    return {
        "records": len(records),
        "unique_videos": unique_video_count,
        "total_views": total_views,
        "average_views": round(average_views, 2),
    }

print("Country summaries:")
for country, records in records_by_country.items():
    # Use the cleaned version of this country's records for accurate summaries.
    country_cleaned = [r for r in cleaned_records if r["country"] == country]
    summary = summarize_country(country_cleaned)
    print(
        f"{country}: records={summary['records']:,}, "
        f"unique videos={summary['unique_videos']:,}, "
        f"average views={summary['average_views']:,.2f}"
    )

Country summaries:
US: records=16,080, unique videos=5,068, average views=1,012,289.20
GB: records=15,075, unique videos=4,113, average views=1,580,015.49
IN: records=15,075, unique videos=5,878, average views=954,592.45


## 9. Final validation

This cell checks the main acceptance requirements and prints a concise run summary. If a requirement fails, fix the file list or the preceding code rather than hiding the failure.

In [10]:
print("=" * 68)
print("TRENDPULSE PART 1 — FINAL VALIDATION")
print("=" * 68)
print(f"Countries: {len(records_by_country)}")
for country, records in records_by_country.items():
    count = sum(1 for record in cleaned_records if record["country"] == country)
    print(f"{country}: {count:,} records")
print(f"Total records: {len(cleaned_records):,}")
print(f"Category lookup entries: {len(category_names):,}")
print(f"Tuple snapshots: {len(video_snapshots):,}")
print(f"Distinct channels: {len(channel_appearances):,}")
print(f"Classification totals: {sum(label_counts.values()):,}")
print("Notebook completed at:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z"))

assert len(records_by_country) >= 3
assert len(cleaned_records) >= 40_000
assert sum(label_counts.values()) == len(cleaned_records)

TRENDPULSE PART 1 — FINAL VALIDATION
Countries: 3
US: 16,080 records
GB: 15,075 records
IN: 15,075 records
Total records: 46,230
Category lookup entries: 17
Tuple snapshots: 46,230
Distinct channels: 438
Classification totals: 46,230
Notebook completed at: 2026-10-27 11:07:34 UTC
